# Análisis Exploratorio de Datos (EDA) - StreamView Analytics
Este notebook forma parte del pipeline de Kedro para el análisis inicial de los datasets proporcionados para el caso de StreamView Analytics.

**Objetivo**:
1. Entender la estructura y calidad de los datos de Películas y Series.
2. Identificar métricas clave de engagement, popularidad y rentabilidad.
3. Perfilar el contenido para el posterior desarrollo del Dashboard Interactivo en React/Plotly.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Configuraciones visuales modernas (modo oscuro para StreamView)
import plotly.io as pio
pio.templates.default = "plotly_dark"


## 1. Carga de Datos
Cargaremos los datasets crudos desde la capa `01_raw` del catálogo de Kedro.


In [ ]:
movies_path = '../data/01_raw/netflix_movies_detailed_up_to_2025.csv'
shows_path = '../data/01_raw/netflix_tv_shows_detailed_up_to_2025.csv'

df_movies = pd.read_csv(movies_path)
df_shows = pd.read_csv(shows_path)

print(f"Total de películas: {df_movies.shape[0]}")
print(f"Total de series: {df_shows.shape[0]}")


## 2. Inspección Inicial y Calidad de Datos


In [ ]:
# Películas
print("--- INFO PELÍCULAS ---")
df_movies.info()

print("\n--- NULOS PELÍCULAS ---")
display(df_movies.isnull().sum())
# NOTA: La columna 'duration' está completamente vacía (16000 nulos).


In [ ]:
# Series
print("--- INFO SERIES ---")
df_shows.info()

print("\n--- NULOS SERIES ---")
display(df_shows.isnull().sum())
# NOTA: 'director' tiene muchísimos nulos. 'duration' sí tiene datos (probablemente nº de temporadas).


## 3. Análisis de Popularidad (Engagement)


In [ ]:
# Distribución de la Popularidad por tipo de contenido
fig = go.Figure()
fig.add_trace(go.Box(y=df_movies['popularity'], name='Películas', marker_color='#1DB954'))
fig.add_trace(go.Box(y=df_shows['popularity'], name='Series', marker_color='#E50914'))

fig.update_layout(
    title="Distribución de la Popularidad (Películas vs Series)",
    yaxis_title="Score de Popularidad",
    yaxis=dict(type='log'), # Escala logarítmica debido a outliers extremos
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.show()


## 4. Evolución de la Plataforma en el Tiempo


In [ ]:
df_movies['date_added'] = pd.to_datetime(df_movies['date_added'], errors='coerce')
df_shows['date_added'] = pd.to_datetime(df_shows['date_added'], errors='coerce')

movies_by_year = df_movies['date_added'].dt.year.value_counts().sort_index()
shows_by_year = df_shows['date_added'].dt.year.value_counts().sort_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=movies_by_year.index, y=movies_by_year.values, mode='lines+markers', name='Películas'))
fig.add_trace(go.Scatter(x=shows_by_year.index, y=shows_by_year.values, mode='lines+markers', name='Series'))

fig.update_layout(
    title="Crecimiento del Catálogo de StreamView por Año",
    xaxis_title="Año de Incorporación",
    yaxis_title="Cantidad de Títulos Añadidos",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.show()


## 5. Acciones a tomar en el Pipeline de Kedro (Capa Intermediate)
1. Rellenar o eliminar la columna `duration` en las películas.
2. Limpiar la columna `date_added` para evitar fechas inválidas.
3. Procesar `genres` para poder analizar por género individual.
